# Real-Time Financial News Sentiment (ABSA)
This notebook fetches current news for a ticker and runs multi-entity ABSA.

In [ ]:
import os, sys
from typing import List, Dict
import pandas as pd

# Ensure the current working directory is in sys.path to find local packages
current_dir = os.path.abspath('.') # Use abspath for consistency, though '.' often works
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)


!sed -i 's/from entity_extraction.ner_extractor import NERExtractor/from ner_extractor import NERExtractor/g' /content/multi_entity_absa.py
!sed -i 's/from entity_extraction.ticker_mapping import map_company_to_ticker/from ticker_mapping import map_company_to_ticker/g' /content/multi_entity_absa.py
!sed -i 's/from models.absa_model import ABSAModel/from absa_model import ABSAModel/g' /content/multi_entity_absa.py


from multi_entity_absa import MultiEntityABSA
from aggregate_sentiment import aggregate_by_ticker
try:
    from yahoo_scraper import fetch_news_for_ticker
    REAL_SCRAPER_AVAILABLE = True
except ImportError:
    REAL_SCRAPER_AVAILABLE = False
    print('[WARN] Could not import fetch_news_for_ticker; using demo articles.')

In [ ]:
def fetch_articles_for_ticker_demo(ticker: str) -> List[Dict]:
    return [
        {
            'id': 'demo-1',
            'text': f"""{ticker} rallied today after strong quarterly earnings. Apple Inc. and Microsoft Corporation also posted gains, while Tesla Inc. slipped on concerns about demand.""",
        },
        {
            'id': 'demo-2',
            'text': """Tesla Inc. shares fell sharply after the company cut prices again, raising worries about profit margins. Meanwhile, Amazon.com Inc. and Microsoft advanced on optimism around cloud and AI.""",
        },
    ]

def fetch_articles_for_ticker_wrapper(ticker: str) -> List[Dict]:
    if REAL_SCRAPER_AVAILABLE:
        print('[INFO] Using real Yahoo scraper.')
        articles = fetch_news_for_ticker(ticker, max_articles=10)
        wrapped = []
        for i, art in enumerate(articles):
            text = art.get('clean_text') or art.get('body') or art.get('summary') or art.get('title') or ''
            if not text:
                continue
            wrapped.append({
                'id': art.get('id', f'art-{i}'),
                'text': text,
                'title': art.get('title', ''),
                'source': art.get('source', ''),
                'published': art.get('published', ''),
            })
        return wrapped
    else:
        print('[INFO] Using demo articles.')
        return fetch_articles_for_ticker_demo(ticker)


In [ ]:
TICKER = 'AAPL'
print('Ticker:', TICKER)
articles = fetch_articles_for_ticker_wrapper(TICKER)
print(f'Fetched {len(articles)} articles.\n')
for art in articles:
    print('ID:', art.get('id'))
    title = art.get('title') or art['text'][:80]
    print('Title/snippet:', title.strip())
    if art.get('published'):
        print('Published:', art['published'])
    print('-' * 80)

In [ ]:
multi_absa = MultiEntityABSA()
records = []
for art in articles:
    art_id = art.get('id', 'unknown')
    text = art['text']
    print(f"\n--- Analyzing article {art_id} ---")
    print(text[:250].strip(), '...\n')
    results = multi_absa.analyze_article(text)
    if not results:
        print('No mapped companies/tickers found in this article.')
        continue
    for r in results:
        rec = {
            'article_id': art_id,
            'company': r['company'],
            'ticker': r['ticker'],
            'sentiment': r['sentiment'],
            'raw_output': r['raw_output'],
        }
        records.append(rec)
        print(
            f"  -> Company: {rec['company']:<25} "
            f"Ticker: {rec['ticker']:<8} "
            f"Sentiment: {rec['sentiment']}"
        )
len(records)

In [ ]:
if not records:
    print('No sentiment records produced.')
else:
    agg = aggregate_by_ticker(records)
    print('\n=== REAL-TIME AGGREGATED SENTIMENT BY TICKER ===')
    for tkr, score in agg.items():
        print(f'Ticker: {tkr:<8}  Score: {score:+.3f}')
    realtime_df = pd.DataFrame(records)
    realtime_df.head()